<a href="https://colab.research.google.com/github/wmjx691/rental-market-analyzer/blob/main/scraper_Google_Maps_Precision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 環境建置與依賴套件 (Environment Setup)
配置爬蟲運行環境，包含 Selenium WebDriver、無頭瀏覽器 (Headless Browser) 設定，以及開源地理編碼套件 `geopy` 的安裝，確保後續網頁渲染與距離計算順利進行。

In [ ]:
# @title 1. 安裝必要套件、瀏覽器驅動與中文字型
!pip install selenium gspread oauth2client webdriver_manager
!pip install geopy googlemaps folium  # 新增 googlemaps 與 folium

# 安裝 Google Chrome 與中文字型
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y
!apt-get install -y fonts-noto-cjk

print("✅ 環境安裝完成！Google Maps API 與 Folium 套件已就緒。")

## 2. Google 雲端服務授權 (GCP Authentication)
建立與 Google Sheets 的安全連線 (OAuth 2.0)。此步驟負責打通資料庫，讓最終清洗完畢的黃金資料能無縫回寫至雲端試算表，實現資料追蹤與自動化更新。

*執行時會跳出視窗要求權限，請點選「允許」。*

In [ ]:
# @title 2. Google 帳號授權與試算表連線
from google.colab import auth
import gspread
from google.auth import default

# 進行身分驗證
auth.authenticate_user()       # 這一行會跳出彈窗要你登入
creds, _ = default()           # 這是暫時性的 Session 憑證
gc = gspread.authorize(creds)

print("Google 帳號授權成功！準備開始爬蟲...")

## 3. 系統參數配置與核心函式 (Config & Core Functions)
定義全域變數以提升程式碼的可維護性與擴展性（Config-Driven）。
使用者可在此自定義**目標行政區**、**物件型態**、**抓取數量上限**以及**距離參考點 (Geo-Fencing 上限)**。同時初始化 Nominatim 地理編碼函式，處理基礎的地址模糊化解析。

In [ ]:
# @title 3. 初始化：全域設定、函式定義與載入資料庫 (Run Once)
import time
import pandas as pd
import re
import pytz
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from geopy.distance import geodesic
import googlemaps
from google.colab import userdata

# ==========================================
# ⚙️ 全域設定區 (CONFIG) - 修改這裡即可改變爬蟲目標
# ==========================================

# 1. 檔案名稱設定 (建議換新名稱以區隔舊格式)
TARGET_DRIVE_FOLDER = 'Demo'       # 指定要存放的資料夾名稱
SHEET_NAME_NEW = 'TW_Rental_Data_Demo_Google_Maps_Precision'

# 2. 地理位置與目標設定
TARGET_CITY = "台北市"                              # 目標縣市 (用於地址解析與補全)
TARGET_REGION_CODE = "1"                          # 地區代碼 (1=台北, 3=新北, 17=高雄, 15=台南...)
TARGET_DISTRICTS = ["中正區", "中山區", "大同區"]    # 目標行政區列表 (可無限新增)
RENTAL_TYPES = ["整層住家", "獨立套房", "分租套房"]  # 選項：整層住家, 獨立套房, 分租套房, 雅房

# 3. 抓取數量上限設定
MAX_ITEMS_PER_TYPE = 100 # 輸入整數，或輸入 'MAX' 抓取全部

# 4. 距離限制設定 (單位：公里)
# 預設大於此距離的物件將被排除。
# 若距離無法計算 (N/A)，則無條件保留供人工確認。
# 若要取消距離限制，請填寫 'MAX'
MAX_DISTANCE_KM = 1

# 5. 參考點設定 (Anchor)
ANCHOR_NAME = "台北車站"
ANCHOR_COORDS = (25.047772,121.516867)

# 6. 其他系統設定
TW_TZ = pytz.timezone('Asia/Taipei')

# ==========================================
# 🔑 載入 Google Maps API Key
# ==========================================
try:
    GMAPS_API_KEY = userdata.get('GMAPS_API_KEY')
    gmaps = googlemaps.Client(key=GMAPS_API_KEY)
    print("✅ Google Maps API 金鑰載入成功！")
except userdata.SecretNotFoundError:
    print("❌ 錯誤：找不到 GMAPS_API_KEY，請確認左側 Secrets 設定。")
# ==========================================

# --- 1. 設定瀏覽器選項 ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# --- 2. 工具函式定義 ---
from googleapiclient.discovery import build # 引入 Drive API 建立工具

def click_element_by_text(driver, text):
    try:
        xpath = f"//label[contains(text(),'{text}')] | //span[contains(text(),'{text}')] | //li[contains(text(),'{text}')]"
        element = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)
        return True
    except: return False

def get_folder_id_by_name_and_parent(folder_name, parent_id=None):
    """輔助函式：根據名稱與父資料夾 ID 尋找或建立單層資料夾"""
    try:
        drive_service = build('drive', 'v3', credentials=creds)
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        if parent_id:
            query += f" and '{parent_id}' in parents"

        response = drive_service.files().list(q=query, spaces='drive', fields='files(id)').execute()
        files = response.get('files', [])

        if files:
            return files[0].get('id')
        else:
            print(f"📁 找不到資料夾 '{folder_name}'，系統自動建立...")
            metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder'}
            if parent_id:
                metadata['parents'] = [parent_id]
            folder = drive_service.files().create(body=metadata, fields='id').execute()
            return folder.get('id')
    except Exception as e:
        print(f"⚠️ Drive API 資料夾操作失敗: {e}")
        return None

def get_or_create_target_folder(path_str):
    """動態解析多層路徑 (支援根目錄、單層、多層) 並回傳最終資料夾 ID"""
    if not path_str or path_str.strip() == "":
        return None

    folder_names = [f.strip() for f in path_str.split('/') if f.strip()]
    current_parent_id = None

    for folder_name in folder_names:
        current_parent_id = get_folder_id_by_name_and_parent(folder_name, current_parent_id)
        if not current_parent_id:
            break

    return current_parent_id

def get_spreadsheet_id_in_folder(sheet_name, folder_id):
    """🆕 新增核心：在指定的資料夾 ID 內，精準尋找特定名稱的試算表 ID"""
    try:
        drive_service = build('drive', 'v3', credentials=creds)
        # 嚴格限制：檔名符合、格式為試算表、未在垃圾桶
        query = f"name='{sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
        if folder_id:
            # 嚴格限制：必須存在於我們剛剛找到的特定資料夾內
            query += f" and '{folder_id}' in parents"

        response = drive_service.files().list(q=query, spaces='drive', fields='files(id)').execute()
        files = response.get('files', [])

        if files:
            return files[0].get('id')
        return None
    except Exception as e:
        print(f"⚠️ 尋找特定資料夾內試算表時發生錯誤: {e}")
        return None

def load_data_from_sheet(sheet_name):
    print(f"📂 正在目標資料夾尋找並讀取工作表: {sheet_name} ...")
    try:
        # 1. 先取得目標資料夾 ID
        folder_id = get_or_create_target_folder(TARGET_DRIVE_FOLDER)

        # 2. 用 API 取得該資料夾內的精準檔案 ID
        file_id = get_spreadsheet_id_in_folder(sheet_name, folder_id)

        if file_id:
            # 3. 使用唯一 ID 開啟檔案，徹底避開全盤搜尋的盲區
            sh = gc.open_by_key(file_id)
        else:
            # 防呆：如果在指定資料夾找不到，嘗試全域搜尋看看是不是放在外面
            sh = gc.open(sheet_name)

        worksheet = sh.sheet1
        data = worksheet.get_all_records()

        if not data:
            print("   -> ⚠️ 檔案存在，但目前內容為空，將從零開始建立資料庫。")
            return {}

        df = pd.DataFrame(data)
        if '物件ID' not in df.columns:
            print(f"   -> ⚠️ 檔案存在，但找不到 '物件ID' 欄位。")
            print(f"   -> 🔍 目前檔案內含有的欄位為: {list(df.columns)}")
            print("   -> ⚠️ 為避免出錯，本次讀取視為空資料庫。若要保留舊資料，請至試算表將 ID 欄位重新命名為 '物件ID'。")
            return {}

        data_map = {}
        for index, row in df.iterrows():
            pid = str(row['物件ID'])
            data_map[pid] = row.to_dict()
        print(f"✅ 成功載入 {len(data_map)} 筆歷史紀錄。")
        return data_map

    except Exception as e:
        print(f"   -> 找不到現有檔案（或讀取失敗），稍後將在存檔時自動建立新表。")
        return {}

def save_full_data(df, sheet_name):
    print(f"💾 正在儲存至: {sheet_name} ...")
    try:
        folder_id = get_or_create_target_folder(TARGET_DRIVE_FOLDER)
        file_id = get_spreadsheet_id_in_folder(sheet_name, folder_id)

        if file_id:
            sh = gc.open_by_key(file_id)
        else:
            print(f"📄 在指定資料夾內找不到現有檔案，正在建立全新試算表...")
            if folder_id:
                sh = gc.create(sheet_name, folder_id=folder_id)
            else:
                sh = gc.create(sheet_name) # 存於根目錄

        worksheet = sh.sheet1
        worksheet.clear()
        worksheet.append_row(df.columns.tolist())
        worksheet.append_rows(df.values.tolist())
        print(f"✅ 儲存完成！共 {len(df)} 筆。連結: {sh.url}")
    except Exception as e:
        print(f"❌ 儲存失敗: {e}")

# --- 3. 升級版地理計算函式 (Google Maps API) ---
def get_distance_from_anchor(address_str):
    """
    輸入地址，透過 Google Maps API 回傳 (距離, 緯度, 經度)。
    """
    if TARGET_CITY not in address_str:
        full_addr = TARGET_CITY + address_str
    else:
        full_addr = address_str

    try:
        # 呼叫 Google Geocoding API
        geocode_result = gmaps.geocode(full_addr)

        if geocode_result:
            # 提取經緯度
            lat = geocode_result[0]['geometry']['location']['lat']
            lng = geocode_result[0]['geometry']['location']['lng']

            target_point = (lat, lng)

            # 使用 geodesic 計算與參考點的直線距離 (km)
            distance = geodesic(ANCHOR_COORDS, target_point).km
            return round(distance, 2), lat, lng
        else:
            return "N/A", "N/A", "N/A"
    except Exception as e:
        print(f"   -> ⚠️ Google Maps API 錯誤: {e}")
        return "N/A", "N/A", "N/A"

# 預設載入資料庫
history_database = load_data_from_sheet(SHEET_NAME_NEW)

## 4-1. 爬蟲階段一：自動化網頁擷取 (Web Scraping Engine)
針對目標租屋網實作的反爬蟲突破機制：
* **多類別輪詢**：自動切換整層住家、獨立/分租套房。
* **無限滾動突破**：利用 DOM 操作與鍵盤模擬 (`Keys.END`) 強制載入動態渲染的物件。
* **唯一 ID 記憶池**：防止「假下一頁按鈕」造成的死迴圈，精準萃取原始網頁文本至記憶體緩存中。

In [ ]:
# @title 4-1. 執行爬蟲：第一階段 (目標租屋網抓取)
if 'history_database' not in globals():
    print("⚠️ 請先執行 Cell 3！")
else:
    limit_text = "無上限" if MAX_ITEMS_PER_TYPE == 'MAX' else f"{MAX_ITEMS_PER_TYPE} 筆"
    dist_text = "無上限" if MAX_DISTANCE_KM == 'MAX' else f"{MAX_DISTANCE_KM} km"
    print(f"🚀 啟動爬蟲 | 目標: {TARGET_CITY} {TARGET_DISTRICTS} | 數量上限: {limit_text} | 距離上限: {dist_text}")

    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.common.action_chains import ActionChains

    chrome_options_optimized = Options()
    for arg in [
        '--headless', '--no-sandbox', '--disable-dev-shm-usage',
        '--window-size=1920,1080', '--disable-blink-features=AutomationControlled'
    ]:
        chrome_options_optimized.add_argument(arg)

    prefs = {"profile.managed_default_content_settings.images": 2, "profile.default_content_setting_values.notifications": 2}
    chrome_options_optimized.add_experimental_option("prefs", prefs)
    chrome_options_optimized.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options_optimized)

    driver.set_page_load_timeout(20)
    raw_data_list = []

    # --- 第一階段：掃描迴圈 ---
    for current_rental_type in RENTAL_TYPES:
        print(f"\n🏎️ [階段一] 正在掃描: {current_rental_type} ...")

        target_url = f"https://rental.example.com.tw/?region={TARGET_REGION_CODE}"

        try:
            driver.get(target_url)
        except Exception:
            print("   ⚠️ 網頁載入超時 (20s)，強制中斷載入並嘗試繼續操作...")
            try: driver.execute_script("window.stop();")
            except: pass

        try:
            WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.close, i.close, .TIGerm"))).click()
        except: pass
        time.sleep(1)

        for district in TARGET_DISTRICTS:
            click_element_by_text(driver, district)

        if not click_element_by_text(driver, current_rental_type):
            print(f"   ⚠️ 無法選取 '{current_rental_type}'，跳過。")
            continue

        try:
            search_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'搜尋')] | //div[contains(@class,'search')]//button")))
            driver.execute_script("arguments[0].click();", search_btn)
        except: pass

        print("   ⏳ 等待列表載入...")
        time.sleep(3)

        page_num = 1
        items_in_this_type = 0
        unique_ids_this_type = set()

        while True:
            # 滾動確保網頁載入
            print(f"      -> 第 {page_num} 頁: 正在向下捲動...", end="\r")
            try:
                body = driver.find_element(By.TAG_NAME, "body")
                for i in range(10):
                    body.send_keys(Keys.END)
                    time.sleep(1.0)
                    next_btns = driver.find_elements(By.XPATH, "//a[contains(@class, 'pageNext')]")
                    if next_btns and next_btns[0].is_displayed():
                        break
            except Exception: pass

            items = driver.find_elements(By.CSS_SELECTOR, ".vue-list-rent-item, .listing-recommend-item, div[class*='item']")

            current_page_new_items = 0
            for item in items:
                try:
                    if item.size['height'] < 80: continue
                    link = "N/A"
                    try: link = item.find_element(By.TAG_NAME, "a").get_attribute("href")
                    except: pass

                    post_id = "N/A"
                    id_match = re.search(r'(\d{7,8})', str(link))
                    if id_match: post_id = id_match.group(1)
                    if post_id == "N/A" or post_id in unique_ids_this_type: continue

                    unique_ids_this_type.add(post_id)

                    raw_data_list.append({
                        "物件ID": post_id,
                        "連結": link,
                        "原始文字": item.text,
                        "物件型態": current_rental_type
                    })
                    current_page_new_items += 1
                except: continue

            items_in_this_type += current_page_new_items
            print(f"      -> 第 {page_num} 頁掃描完畢: 本頁新增 {current_page_new_items} 筆 (累積 {items_in_this_type} 筆)           ", end="\r")

            if MAX_ITEMS_PER_TYPE != 'MAX' and isinstance(MAX_ITEMS_PER_TYPE, int):
                if items_in_this_type >= MAX_ITEMS_PER_TYPE:
                    print(f"\n      🛑 已達到設定的抓取上限 ({MAX_ITEMS_PER_TYPE} 筆)，停止翻頁。")
                    break

            if current_page_new_items == 0:
                print(f"\n      🛑 本頁未發現新物件，停止翻頁。")
                break

            try:
                next_btns = driver.find_elements(By.XPATH, "//a[contains(@class, 'pageNext') or contains(text(), '下一頁')]")
                if not next_btns: break

                next_btn = next_btns[0]
                btn_class = next_btn.get_attribute("class") or ""
                if "disabled" in btn_class or "last" in btn_class: break

                driver.execute_script("arguments[0].click();", next_btn)
                page_num += 1
                time.sleep(3)

            except Exception as e: break

        print(f"\n   ✅ {current_rental_type} 掃描完成。")

    driver.quit()

## 4-2. 爬蟲階段二：資料清洗與地理圍欄 (Data Pipeline & Geo-Fencing)
讀取階段一的記憶體快取，進行二次處理：
* **正規化萃取**：利用正則表達式 (Regex) 精準抓取價格、坪數並進行極端值過濾。
* **狀態即時監控**：導入 `clear_output` 實現單點即時進度刷新，視覺化監控 API 請求狀態。
* **開源地理運算**：使用 Nominatim API 呼叫地址經緯度，並以 Haversine 公式計算物件與參考點的距離，剔除超標物件後，合併歷史資料匯出。

In [ ]:
# @title 4-2. 執行爬蟲：第二階段 (資料處理與地理距離計算) - 🔄 單筆刷新版
from IPython.display import clear_output
import time
from datetime import datetime
import pandas as pd
import re

if 'raw_data_list' not in globals() or not raw_data_list:
    print("⚠️ 找不到 raw_data_list，請先執行 4-1 完成第一階段掃描！")
else:
    processed_data_list = []
    today_str = datetime.now(TW_TZ).strftime("%Y-%m-%d")
    last_check_time = datetime.now(TW_TZ).strftime("%Y-%m-%d %H:%M:%S")
    district_regex_str = f"({'|'.join(TARGET_DISTRICTS)})"

    unique_raw_data = {item['物件ID']: item for item in raw_data_list}.values()
    total_items = len(unique_raw_data)
    processed_count = 0
    filtered_by_distance_count = 0

    for raw in unique_raw_data:
        processed_count += 1

        try:
            post_id = raw['物件ID']
            full_text = raw['原始文字']
            link = raw['連結']
            item_type = raw['物件型態']

            # --- 🖥️ 畫面刷新與狀態標頭 ---
            clear_output(wait=True)
            print(f"⚙️ [階段二] 開始處理 {len(raw_data_list)} 筆原始資料 (解析 + 算距離)...\n")
            print(f"進度: {processed_count}/{total_items} | 物件ID: {post_id}")
            text_snippet = full_text.replace('\n', ' ')[:40]
            print(f"   -> 原始文字片段: {text_snippet}...")

            if len(full_text) < 10:
                print("   -> ⚠️ 文字過短，跳過。")
                time.sleep(0.3)
                continue

            price = "N/A"
            match = re.search(r'(\d{1,3}(,\d{3})*)\s*元/月', full_text)
            if match: price = match.group(0).replace("元/月", "").strip()
            else:
                print("   -> ⚠️ 找不到價格，跳過。")
                time.sleep(0.3)
                continue

            area = "N/A"
            match = re.search(r'(\d+\.?\d*)\s*坪', full_text)
            if match: area = match.group(1)

            try:
                if area != "N/A":
                    area_f = float(area)
                    if item_type == "整層住家" and area_f < 15: continue
                    if "套房" in item_type and area_f < 4: continue
            except: pass

            lines = full_text.split('\n')
            title = lines[0] if lines else "N/A"
            if len(title) < 5 and len(lines) > 1: title = lines[1]

            site_update_text = "N/A"
            for line in lines:
                if any(k in line for k in ["更新", "發佈", "前", "昨天", "剛"]):
                    if len(line) < 15: site_update_text = line; break

            location_full, city, district, address_road = "N/A", TARGET_CITY, "N/A", "N/A"
            for line in lines:
                if "區" in line and ("市" in line or "-" in line):
                    location_full = line; break
            if location_full != "N/A":
                parts = re.split(r'[-/ ]', location_full)
                for p in parts:
                    if "區" in p: district = p
                    if "路" in p or "街" in p: address_road = p
            if district == "N/A":
                m = re.search(district_regex_str, full_text)
                if m: district = m.group(1)

            dist_km, lat, lon = "N/A", "N/A", "N/A"
            is_processed = False

            if post_id in history_database:
                old = history_database[post_id]
                if '距離(km)' in old and old['距離(km)'] != "N/A":
                    dist_km = old['距離(km)']
                    lat = old.get('緯度', "N/A")
                    lon = old.get('經度', "N/A")
                    is_processed = True
                    print(f"   -> ♻️ 歷史資料已存在，直接沿用距離: {dist_km} km")

            if not is_processed and address_road != "N/A":
                search_target = location_full if location_full != "N/A" else f"{district}{address_road}"
                search_target = search_target.replace("-", "").replace("/", "")

                print(f"   -> 📡 準備呼叫 API 解析地址: {search_target} ...", end=" ")
                dist_km, lat, lon = get_distance_from_anchor(search_target)
                print(f"✅ 完成！距離: {dist_km} km")

                # 改為 Google Maps API 的高併發設定，保留微小延遲讓畫面順暢
                time.sleep(0.1)

            # --- 距離過濾邏輯 ---
            if MAX_DISTANCE_KM != 'MAX' and isinstance(MAX_DISTANCE_KM, (int, float)):
                if dist_km != "N/A":
                    try:
                        if float(dist_km) > MAX_DISTANCE_KM:
                            filtered_by_distance_count += 1
                            print(f"   -> 🛑 距離 {dist_km} km 超過上限 {MAX_DISTANCE_KM} km，已排除。")
                            time.sleep(0.5) # 稍微暫停讓肉眼看到排除訊息
                            continue
                    except: pass

            # --- 紀錄存活的天數 ---
            first_seen, days_mkt = today_str, 0
            if post_id in history_database:
                old = history_database[post_id]
                if '首次發現日' in old and old['首次發現日']:
                    first_seen = old['首次發現日']
                    try: days_mkt = (datetime.strptime(today_str, "%Y-%m-%d") - datetime.strptime(first_seen, "%Y-%m-%d")).days
                    except: pass

            processed_data_list.append({
                "物件ID": post_id,
                "最後更新時間": last_check_time,
                "首次發現日": first_seen,
                "上架已持續天數": days_mkt,
                "網站顯示更新": site_update_text,
                "物件型態": item_type,
                "標題": title,
                "價格": price,
                "坪數": area,
                "距離(km)": dist_km,
                "緯度": lat,
                "經度": lon,
                "完整顯示地址": location_full,
                "路段/地址": address_road,
                "連結": link
            })
            print("   -> 📥 成功加入處理清單。")

            # 若不是呼叫 API (歷史沿用)，稍微停頓一下讓畫面不會閃爍過快
            if is_processed: time.sleep(0.1)

        except Exception as e:
            print(f"   -> ❌ 發生未預期錯誤: {e}")
            time.sleep(1)
            continue

    # --- 迴圈結束，顯示最終統計結果 ---
    clear_output(wait=True)
    print(f"\n📊 統計: 處理完成，共產出 {len(processed_data_list)} 筆有效資料 (因距離過遠排除 {filtered_by_distance_count} 筆)，開始存檔...")
    df_new = pd.DataFrame(processed_data_list)
    if not df_new.empty:
        df_new['廣告投放數'] = df_new.groupby(['價格', '坪數'])['物件ID'].transform('count')

        final_map = history_database.copy()
        for idx, row in df_new.iterrows():
            final_map[row['物件ID']] = row.to_dict()

        df_final = pd.DataFrame(list(final_map.values()))
        if '最後更新時間' in df_final.columns:
            df_final = df_final.sort_values(by='最後更新時間', ascending=False)

        cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "物件型態", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]
        df_final = df_final[[c for c in cols if c in df_final.columns]]

        save_full_data(df_final, SHEET_NAME_NEW)
        history_database = final_map
    else:
         print("⚠️ 過濾後沒有剩餘資料可供存檔。")

In [ ]:
# @title 5. 地圖視覺化與自動存檔 (Drive API 無縫上傳版)
import folium
import pandas as pd
import math
import os
from datetime import datetime
from IPython.display import display
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

print(f"🗺️ 正在從 {SHEET_NAME_NEW} 載入資料以生成地圖...")

try:
    # 1. 從剛存檔的 Google Sheet 讀取資料
    sh = gc.open(SHEET_NAME_NEW)
    worksheet = sh.sheet1
    data = worksheet.get_all_records()
    df_map = pd.DataFrame(data)

    if df_map.empty:
        print("⚠️ 警告：目前資料庫是空的，沒有資料可以畫在地圖上。")
    else:
        # 2. 建立地圖
        m = folium.Map(location=[ANCHOR_COORDS[0], ANCHOR_COORDS[1]], zoom_start=15)

        # 標記中心點
        folium.Marker(
            [ANCHOR_COORDS[0], ANCHOR_COORDS[1]],
            popup=f"<b>{ANCHOR_NAME}</b>",
            icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(m)

        # 3. 標記物件
        valid_count = 0
        for idx, row in df_map.iterrows():
            try:
                lat, lng = float(row['緯度']), float(row['經度'])
                if math.isnan(lat) or math.isnan(lng): continue

                icon_color = "green" if "套房" in str(row.get('物件型態', '')) else "orange"
                popup_html = f"""<div style="min-width: 150px;">
                    <b><a href="{row.get('連結','#')}" target="_blank">{row.get('標題','無')}</a></b><br>
                    租金: {row.get('價格','?')} | 坪數: {row.get('坪數','?')}<br>
                    距離: {row.get('距離(km)','?')} km</div>"""

                folium.Marker(
                    [lat, lng],
                    popup=folium.Popup(popup_html, max_width=300),
                    icon=folium.Icon(color=icon_color, icon="home")
                ).add_to(m)
                valid_count += 1
            except: continue

        print(f"✅ 地圖生成成功！共標記了 {valid_count} 個物件。")

        # 4. 先將地圖存為 Colab 本地暫存檔
        current_time = datetime.now(TW_TZ).strftime("%Y%m%d_%H%M")
        file_name = f"rental_map_{current_time}.html"
        local_path = f"/content/{file_name}"
        m.save(local_path)

        # ==========================================
        # 🚀 5. 透過 Drive API 背景無縫上傳
        # 直接沿用 Cell 2 取得的全域憑證 `creds`
        # ==========================================
        print("☁️ 正在透過 Drive API 背景上傳至指定資料夾...")
        drive_service = build('drive', 'v3', credentials=creds)

# 輔助函式 1：透過資料夾名稱尋找 ID (若無則自動建立)
        def get_folder_id(folder_name, parent_id=None):
            query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
            if parent_id:
                query += f" and '{parent_id}' in parents"
            response = drive_service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
            files = response.get('files', [])

            if files:
                return files[0].get('id')
            else:
                # 找不到，自動建立資料夾
                metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder'}
                if parent_id:
                    metadata['parents'] = [parent_id]
                folder = drive_service.files().create(body=metadata, fields='id').execute()
                return folder.get('id')

        # 輔助函式 2：動態解析並建立多層路徑
        def get_or_create_path(path_str):
            # 移除前後空白，並用 "/" 切割字串，過濾掉多餘的空字串 (例如處理 "A//B" 的防呆)
            folder_names = [f.strip() for f in path_str.split('/') if f.strip()]
            current_parent_id = None

            for folder_name in folder_names:
                # 迭代尋找或建立每一層資料夾，並將 ID 傳遞給下一層當作 parent_id
                current_parent_id = get_folder_id(folder_name, current_parent_id)

            return current_parent_id

        # ==========================================
        # 🎯 定義目標路徑並取得最終 ID
        # ==========================================
        TARGET_DRIVE_PATH = 'map'
        print(f"📂 正在鎖定或建立目標路徑: {TARGET_DRIVE_PATH} ...")

        target_folder_id = get_or_create_path(TARGET_DRIVE_PATH)

        # 執行檔案上傳
        file_metadata = {
            'name': file_name,
            'parents': [target_folder_id]
        }

        media = MediaFileUpload(local_path, mimetype='text/html')
        uploaded_file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

        print(f"🎉 檔案已成功存入雲端硬碟！(完全免二次授權)")

        # 在 Colab 輸出區塊直接顯示互動地圖
        display(m)

except Exception as e:
    print(f"❌ 生成或上傳地圖時發生錯誤: {e}")

---

### 如何執行你的「一週測試計畫」

既然你要測試一週，且每三天執行一次，操作流程如下：

1. **第一次（今天）：**
* 登入你的 Google Drive，建立一個 Colab 筆記本。
* 將上述三段代碼貼入。
* 依序點擊「播放鍵」執行 Cell 1, 2, 3, 4-1, 4-2。
* 執行完後，去你的 Google Drive 根目錄找找看，會有一個你在CONFIG設定中名為 **`SHEET_NAME_NEW`** 的試算表。打開來確認資料是否正確。


2. **第二次（三天後）：**
* 打開這個 Colab 網頁。
* **重要：** 因為 Colab 會重置環境，所以你必須**再次點擊 Cell 1, 2, 3, 4-1, 4-2**。
* 程式會自動把新的資料「新增」到那張試算表的下面，不會覆蓋舊資料。


3. **第三次（六天後）：**
* 重複上述動作。



### 提醒（關於目標租屋網的反爬蟲）

在 Colab 的「無頭模式（Headless）」下，瀏覽器特徵非常明顯，租屋網這種網站有時會直接阻擋（你可能會看到程式跑完但說「抓到 0 筆物件」）。

* **如果發生這種情況**：代表目標租屋網擋掉了 Colab 的 IP 或特徵。這時候最簡單的解法，還是回到我一開始提供的 **PC 本地端執行**（因為你在本地有視窗介面，比較像真人）。

#### 第五步(Optional)：一次性資料遷移腳本 (One-time Migration)（Cell 5）

它會讀取 SHEET_NAME_OLD (備份檔)，針對每一筆資料檢查是否缺少距離。

如果缺少，呼叫我們新的強效地理計算函式補上，最後將補完的資料存入 SHEET_NAME_NEW。

執行完這次後，這個 Cell 就不需要再跑了。

In [ ]:
# @title 🛠️ 一次性維護：歷史資料全面升級與清洗 (Google Maps API)
import pandas as pd
import time
from IPython.display import clear_output

print(f"📥 讀取目前資料庫: {SHEET_NAME_NEW} ...")
try:
    sh = gc.open(SHEET_NAME_NEW)
    worksheet = sh.sheet1
    data = worksheet.get_all_records()
    df = pd.DataFrame(data)

    if df.empty:
        print("資料庫是空的，無需清洗。")
    else:
        print(f"共載入 {len(df)} 筆歷史資料，準備強制重新計算距離...")
        valid_records = []
        filtered_count = 0
        error_count = 0

        for index, row in df.iterrows():
            post_id = row.get('物件ID', '未知')
            address = row.get('完整顯示地址', '')
            if not address or address == "N/A":
                address = row.get('路段/地址', '')

            clear_output(wait=True)
            print(f"⚙️ 歷史資料升級中: {index+1}/{len(df)}")
            print(f"   -> 處理 ID: {post_id} | 地址: {address}")

            # 強制呼叫 Google Maps API
            dist_km, lat, lng = get_distance_from_anchor(str(address))

            # Google API 容許高併發，不需要像開源 OSM 停頓 1.5 秒，但稍微停頓 0.05 秒讓畫面順暢
            time.sleep(0.05)

            if dist_km != "N/A":
                try:
                    dist_val = float(dist_km)
                    if MAX_DISTANCE_KM != 'MAX' and dist_val > float(MAX_DISTANCE_KM):
                        print(f"   -> 🛑 距離 {dist_km} km 超過 {MAX_DISTANCE_KM} km，捨棄！")
                        filtered_count += 1
                        continue
                except:
                    pass

            # 更新為 Google 經緯度與新距離
            row['距離(km)'] = dist_km
            row['緯度'] = lat
            row['經度'] = lng
            valid_records.append(row)

        # 儲存覆寫回 Google Sheets
        clear_output(wait=True)
        print(f"📊 清洗完成統計：")
        print(f"   - 原始資料: {len(df)} 筆")
        print(f"   - 成功保留: {len(valid_records)} 筆 (均在 {MAX_DISTANCE_KM} km 內)")
        print(f"   - 超界剔除: {filtered_count} 筆")

        df_clean = pd.DataFrame(valid_records)
        save_full_data(df_clean, SHEET_NAME_NEW)

        # 同步更新記憶體中的歷史資料庫，以免等一下跑爬蟲又錯亂
        history_database = load_data_from_sheet(SHEET_NAME_NEW)
        print("🎉 歷史資料庫全面升級為 Google Maps 座標系統完畢！")

except Exception as e:
    print(f"❌ 讀取或處理時發生錯誤: {e}")

In [ ]:
# @title 6. 資料遷移：補完舊資料的經緯度 (執行一次即可)

def migrate_old_data():
    print("🚀 開始執行舊資料遷移與補完計畫...")

    # 1. 讀取舊資料 (備份檔)
    old_map = load_data_from_sheet(SHEET_NAME_OLD)
    if not old_map:
        print("❌ 找不到舊資料備份，請確認 SHEET_NAME_OLD 設定是否正確。")
        return

    print(f"📦 讀取到 {len(old_map)} 筆舊資料，開始檢查缺失的經緯度...")

    updated_count = 0
    processed_list = []

    # 2. 遍歷舊資料
    for pid, row in old_map.items():
        # 檢查是否需要補距離
        need_update = False
        dist = str(row.get('距離(km)', 'N/A'))

        # 如果距離是 N/A 或者根本沒有這個欄位，就需要補算
        if dist == 'N/A' or dist == '':
            addr_full = str(row.get('完整顯示地址', ''))
            addr_road = str(row.get('路段/地址', ''))

            # 組合搜尋字串
            search_target = addr_full if len(addr_full) > 5 else f"高雄市{addr_road}"
            search_target = search_target.replace("-", "").replace("/", "")

            if len(search_target) > 3: # 確保有字可查
                print(f"   🔧 正在補算: {row.get('標題', 'Unknown')} ({search_target})...")
                dist_km, lat, lon = get_distance_from_anchor(search_target)

                # 更新欄位
                row['距離(km)'] = dist_km
                row['緯度'] = lat
                row['經度'] = lon

                updated_count += 1
                time.sleep(1.1) # 禮貌性延遲
            else:
                row['距離(km)'] = 'N/A'
                row['緯度'] = 'N/A'
                row['經度'] = 'N/A'

        processed_list.append(row)

    print(f"✨ 補完作業結束！共更新了 {updated_count} 筆資料。")

    # 3. 讀取新表目前的資料 (如果有的話)，避免覆蓋掉剛剛爬蟲剛抓的新資料
    #    但如果你的目的是「用舊資料初始化新表」，則直接存入即可。
    #    這裡我們採用「合併」策略：舊資料補完後 + 新表已有的資料

    new_map_current = load_data_from_sheet(SHEET_NAME_NEW)

    # 將補完的舊資料 merge 進去 (如果 ID 相同，以舊資料為主，因為我們要保留首次發現日)
    # 但通常我們希望保留最新的「最後更新時間」。
    # 策略：以 ID 為準，將 processed_list 轉回 map

    final_map = new_map_current.copy()
    for row in processed_list:
        pid = str(row['物件ID'])
        # 如果新表中已經有這筆，我們只更新它的經緯度 (避免覆蓋掉最新的價格或狀態)
        if pid in final_map:
            if row['距離(km)'] != 'N/A':
                final_map[pid]['距離(km)'] = row['距離(km)']
                final_map[pid]['緯度'] = row['緯度']
                final_map[pid]['經度'] = row['經度']
        else:
            # 如果新表沒這筆，直接加入
            final_map[pid] = row

    # 4. 存回新表
    df_final = pd.DataFrame(list(final_map.values()))

    # 排序
    if '最後更新時間' in df_final.columns:
        df_final = df_final.sort_values(by='最後更新時間', ascending=False)

    # 欄位整理
    cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]
    df_final = df_final[[c for c in cols if c in df_final.columns]]

    save_full_data(df_final, SHEET_NAME_NEW)

    # 更新全域變數，讓爬蟲知道
    global history_database
    history_database = final_map

# 執行遷移
migrate_old_data()

In [ ]:
# @title 6. 資料遷移：將舊資料格式升級並補上「物件型態」 (Run Once)

def migrate_schema_add_type():
    print("🚀 開始執行資料庫升級遷移 (Schema Migration)...")

    # 1. 讀取舊資料 (來源)
    # 這裡我們讀取你在 Cell 3 設定的 SHEET_NAME_OLD
    old_map = load_data_from_sheet(SHEET_NAME_OLD)

    if not old_map:
        print(f"❌ 找不到舊資料來源 ({SHEET_NAME_OLD})，無法進行遷移。")
        return

    print(f"📦 讀取到 {len(old_map)} 筆舊資料，開始進行格式轉換...")

    migrated_list = []

    # 2. 遍歷舊資料並補全欄位
    for pid, row in old_map.items():
        # A. 補上「物件型態」
        # 因為上一版爬蟲只抓整層住家，所以舊資料全部標記為 "整層住家"
        if '物件型態' not in row or not row['物件型態']:
            row['物件型態'] = "整層住家"

        # B. 確保其他新欄位存在 (防呆)
        # 如果舊資料還沒補過距離，這裡補上預設值 N/A (避免存檔報錯)
        # (如果你之前跑過 Cell 5 補距離，這裡會保留原本的值)
        if '距離(km)' not in row: row['距離(km)'] = "N/A"
        if '緯度' not in row: row['緯度'] = "N/A"
        if '經度' not in row: row['經度'] = "N/A"
        if '網站顯示更新' not in row: row['網站顯示更新'] = "N/A"

        migrated_list.append(row)

    # 3. 轉為 DataFrame 並存入「新表」
    df_migrated = pd.DataFrame(migrated_list)

    # 排序 (如果有的話)
    if '最後更新時間' in df_migrated.columns:
        df_migrated = df_migrated.sort_values(by='最後更新時間', ascending=False)

    # 定義新版的完整欄位順序
    cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "物件型態", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]

    # 過濾掉不存在的欄位 (防呆) 並重組順序
    final_cols = [c for c in cols if c in df_migrated.columns]
    df_migrated = df_migrated[final_cols]

    print(f"💾 正在將 {len(df_migrated)} 筆升級後的資料寫入新表: {SHEET_NAME_NEW} ...")
    save_full_data(df_migrated, SHEET_NAME_NEW)

    # 4. 更新記憶體中的全域變數
    # 這樣你不用重跑 Cell 3，等一下直接跑 Cell 4 爬蟲就會以這份新資料為基礎
    global history_database

    # 將 list 轉回 map 格式更新 history_database
    new_history_map = {}
    for index, row in df_migrated.iterrows():
        pid = str(row['物件ID'])
        new_history_map[pid] = row.to_dict()

    history_database = new_history_map
    print("✨ 資料庫升級完成！你可以開始執行 Cell 4 進行多型態爬蟲了。")

# 執行遷移
migrate_schema_add_type()